# Measurements explorer

Load MIRICO Orion measurements for one or more deployments and work with them through a small `Deployment` object:

| attribute / method | what you get |
|---|---|
| `d.report` | every reading (one row per retro measurement) |
| `d.retros` | retro names |
| `d.retro('R4')` | readings for one retro, time-indexed, with CH$_4$ in ppm |
| `d.ch4(rate)` / `d.ch4(rate, stderr=True)` | wide table, one column per retro (ppm), optionally resampled |
| `d.sensors` | Orion / retro coordinates (LLA) and retro ENU offsets from the Orion (m) |
| `d.wind` | high-res wind if loaded, else the low-res wind from the report |
| `d.summary()` | per-retro counts, time span and CH$_4$ stats |
| `d.obs_targets(rate)` | binned targets in the `{'R4obs0': {'val', 'sigma'}}` form used by the history-matching notebook |
| `d.data` / `d.dataset` | the underlying pyelq `MiricoData` / raw `xarray.Dataset` |

Downloaded datasets are cached under `output/measurements_cache/`, so re-running is quick. Pass `refresh=True` to re-download.

## 1. Credentials
Pick the env file. `.env_durham` is the limited read-only set intended for sharing; `.env` is the full internal set.

Any `AZURE_STORAGE_*` / `MIRICO_*` variables already in the environment are cleared first, so what you test is exactly what is in `ENV_FILE` (nothing leaks in from a previously loaded `.env`).

In [ ]:
import json
import os
import pickle
import sys
from dataclasses import dataclass
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import parse_qs
from urllib.request import urlopen

import numpy as np
import pandas as pd
from dotenv import load_dotenv

ENV_FILE = Path('.env_durham')   # or Path('.env')

# Allow running from the repo root as well as from mirico/
if not ENV_FILE.exists() and (Path('mirico') / ENV_FILE).exists():
    ENV_FILE = Path('mirico') / ENV_FILE
if not ENV_FILE.exists():
    raise FileNotFoundError(f'{ENV_FILE} not found (cwd = {Path.cwd()})')

# Make pyelq importable when it is a local checkout rather than an installed package
mipyelq_src = Path('/Users/hamid/mirico/mipyelq/src')
if mipyelq_src.exists() and str(mipyelq_src) not in sys.path:
    sys.path.insert(0, str(mipyelq_src))

for key in [k for k in os.environ if k.startswith(('AZURE_STORAGE_', 'MIRICO_'))]:
    del os.environ[key]
load_dotenv(ENV_FILE, override=True)
print('Loaded credentials from', ENV_FILE.resolve())

In [ ]:
def sas_fields(token):
    '''Parse the non-secret fields of a SAS token (scope, permissions, expiry).'''
    return {k: v[0] for k, v in parse_qs(token).items() if k != 'sig'}


def describe_credentials():
    '''Table of the loaded variables. Token signatures are never shown.'''
    now = pd.Timestamp.now(tz='UTC')
    rows = []
    for key in sorted(k for k in os.environ if k.startswith(('AZURE_STORAGE_', 'MIRICO_'))):
        value = os.environ[key]
        row = {'variable': key}
        if 'SAS_TOKEN' in key:
            f = sas_fields(value)
            expires = pd.Timestamp(f['se']) if 'se' in f else pd.NaT
            row.update(scope={'c': 'container', 'b': 'blob'}.get(f.get('sr'), f.get('sr')),
                       permissions=f.get('sp'), expires=expires,
                       valid_now=bool(pd.notna(expires) and expires > now))
        else:
            row['value'] = value
        rows.append(row)
    return pd.DataFrame(rows).set_index('variable')


describe_credentials()

### Access check
Lists a few blobs from each instrument's container with its SAS token, straight against the Azure REST API (no pyblobtools needed). `OK` means the token works for that container; `AuthenticationFailed` means the token was not signed for that account/container (or the signing key has been rotated).

Container names come from the instrument index (`index/zarr_index.json`) on the main storage account. If the shared account uses different container names, edit `CONTAINERS`.

In [ ]:
ACCOUNT = os.environ.get('AZURE_STORAGE_ACCOUNT')
CONTAINERS = {'NG5': 'ng5-nuc11tnhi7', 'NG6': 'ng6-nuc11tnhi7'}


def _xml_tag(body, tag):
    return [part.split(f'</{tag}>')[0] for part in body.split(f'<{tag}>')[1:]]


def check_access(machine, container=None, n=5):
    container = container or CONTAINERS[machine]
    token = os.environ.get(f'AZURE_STORAGE_SAS_TOKEN_{machine}')
    if not token:
        return f'{machine}: MISSING  no AZURE_STORAGE_SAS_TOKEN_{machine} in {ENV_FILE}'
    url = (f'https://{ACCOUNT}.blob.core.windows.net/{container}'
           f'?restype=container&comp=list&delimiter=/&maxresults={n}&{token}')
    try:
        with urlopen(url, timeout=30) as resp:
            body = resp.read().decode()
    except HTTPError as err:
        reason = (_xml_tag(err.read().decode(), 'Code') or [err.reason])[0]
        return f'{machine}: FAILED  HTTP {err.code} {reason}  ({ACCOUNT}/{container})'
    return f'{machine}: OK  {ACCOUNT}/{container} -> {_xml_tag(body, "Name")}'


for machine in CONTAINERS:
    print(check_access(machine))

# pyblobtools.load_zarr looks deployments up in the index before reading data.
print('index token present:', 'AZURE_STORAGE_SAS_TOKEN_INDEX' in os.environ)

### (Optional) What deployments exist?
Reads `zarr_index.json` from the `index` container. Needs `AZURE_STORAGE_SAS_TOKEN_INDEX`, which `.env_durham` does not currently have.

In [ ]:
def list_index_deployments(machine):
    token = os.environ.get('AZURE_STORAGE_SAS_TOKEN_INDEX')
    if not token:
        print(f'No AZURE_STORAGE_SAS_TOKEN_INDEX in {ENV_FILE}: cannot read the index.')
        return None
    with urlopen(f'https://{ACCOUNT}.blob.core.windows.net/index/zarr_index.json?{token}', timeout=60) as resp:
        index = json.load(resp)
    df = pd.DataFrame(index.get(machine.lower(), []))
    if df.empty:
        return df
    df[['startDate', 'endDate']] = df[['startDate', 'endDate']].apply(pd.to_datetime)
    return df[['project', 'startDate', 'endDate', 'gpsLat', 'gpsLong']].sort_values('startDate')


list_index_deployments('NG5')

## 2. Deployments
One entry per deployment / time window. The key is just a label you choose; use it to pick the deployment later. Add as many as you like.

In [ ]:
DEPLOYMENTS = {
    'ng5_2025-06-18': dict(machine='NG5', start='2025-06-18 14:36:00', end='2025-06-18 15:30:00'),
    # 'ng6_example': dict(machine='NG6', start='2025-09-12 10:00:00', end='2025-09-12 12:00:00'),
}

EXTRA_LOADERS = ['hf_wind', 'lf_wind', 'settings', 'environment']
CACHE_DIR = Path('output/measurements_cache')

In [ ]:
import warnings
warnings.simplefilter('ignore', category=FutureWarning)

import pyblobtools as pbt
from pyelq.coordinate_system import LLA
from pyelq.data_access.xarray import load_from_xarray


def _decode(value):
    return value.decode('utf-8') if isinstance(value, bytes) else value


@dataclass
class Deployment:
    name: str
    machine: str
    start: pd.Timestamp
    end: pd.Timestamp
    dataset: object   # raw xarray.Dataset from pyblobtools
    data: object      # pyelq MiricoData built from it

    @property
    def report(self):
        '''Every reading: one row per retro measurement (ch4 in mol/mol).'''
        return self.data.report

    @property
    def settings(self):
        return self.data.settings

    @property
    def retros(self):
        return sorted(self.report['retro_name'].unique())

    @property
    def sensors(self):
        '''Orion / retro positions per beam, plus retro ENU offsets (m) from the first Orion.'''
        rows = []
        for retro_id, s in self.settings.items():
            (o_lat, o_lon, o_alt), (r_lat, r_lon, r_alt) = s['orion_lla'], s['retro_lla']
            rows.append(dict(retro_name=_decode(s.get('retro_name', retro_id)), retro_id=retro_id,
                             orion_lat=o_lat, orion_lon=o_lon, orion_alt=o_alt,
                             retro_lat=r_lat, retro_lon=r_lon, retro_alt=r_alt))
        df = pd.DataFrame(rows).set_index('retro_name').sort_index()
        ref = df.iloc[0]
        enu = LLA(latitude=df['retro_lat'].to_numpy(), longitude=df['retro_lon'].to_numpy(),
                  altitude=df['retro_alt'].to_numpy()).to_enu(
            ref_latitude=ref['orion_lat'], ref_longitude=ref['orion_lon'], ref_altitude=ref['orion_alt'])
        df['east'], df['north'], df['up'] = enu.east, enu.north, enu.up
        return df

    @property
    def wind(self):
        '''High-res wind if it was loaded, otherwise the low-res wind columns of the report.'''
        hr = getattr(self.data, 'high_res_wind', None)
        if hr is not None and len(hr):
            tcol = 'windtime' if 'windtime' in hr.columns else 'timestamp'
            return hr.set_index(tcol).sort_index()
        cols = [c for c in ('windx', 'windy', 'windz') if c in self.report.columns]
        return self.report.set_index('timestamp')[cols].sort_index()

    def retro(self, name):
        '''Readings for one retro, indexed by time, with CH4 in ppm added.'''
        df = self.report[self.report['retro_name'] == name].set_index('timestamp').sort_index()
        return df.assign(ch4_ppm=df['ch4'] * 1e6, ch4_stderr_ppm=df['ch4_stderr'] * 1e6)

    def ch4(self, rate=None, stderr=False):
        '''Wide table, one column per retro (ppm). rate e.g. "60s" or "300s" resamples to bin means.
        Unresampled, the table is mostly NaN because the Orion visits retros in turn.'''
        col = 'ch4_stderr' if stderr else 'ch4'
        wide = self.report.pivot_table(index='timestamp', columns='retro_name', values=col) * 1e6
        return wide.resample(rate).mean() if rate else wide

    def obs_targets(self, rate='300s'):
        '''Binned targets keyed like the history-matching notebook: {"R4obs0": {"val": ppm, "sigma": ppm}}.'''
        mean, sd = self.ch4(rate), self.ch4(rate, stderr=True)
        targets = {}
        for retro in mean.columns:
            for i, (m, s) in enumerate(zip(mean[retro], sd[retro])):
                if pd.notna(m) and pd.notna(s):
                    targets[f'{retro}obs{i}'] = {'val': m, 'sigma': s}
        return targets

    def summary(self):
        g = self.report.groupby('retro_name')
        return pd.DataFrame({
            'n_readings': g.size(),
            'first': g['timestamp'].min(),
            'last': g['timestamp'].max(),
            'ch4_mean_ppm': g['ch4'].mean() * 1e6,
            'ch4_max_ppm': g['ch4'].max() * 1e6,
            'stderr_mean_ppm': g['ch4_stderr'].mean() * 1e6,
        })

    def __repr__(self):
        return (f'<Deployment {self.name}: {self.machine} {self.start} -> {self.end}, '
                f'{len(self.report)} readings, retros={self.retros}>')


def load_deployment(name, refresh=False):
    '''Load a deployment from DEPLOYMENTS, using the local cache unless refresh=True.'''
    cfg = DEPLOYMENTS[name]
    cache = CACHE_DIR / f'{name}.pkl'
    if cache.exists() and not refresh:
        with open(cache, 'rb') as fh:
            dataset = pickle.load(fh)
    else:
        dataset = pbt.load_zarr(cfg['machine'], cfg['start'], cfg['end'],
                                extra_loaders=cfg.get('extra_loaders', EXTRA_LOADERS)).load()
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        with open(cache, 'wb') as fh:
            pickle.dump(dataset, fh)
    data = load_from_xarray(dataset, post_calculate_diffusion_params=True)
    data.report['retro_name'] = data.report['retro_name'].apply(_decode)
    data.report['timestamp'] = pd.to_datetime(data.report['timestamp'])
    return Deployment(name, cfg['machine'], pd.Timestamp(cfg['start']), pd.Timestamp(cfg['end']),
                      dataset, data)

## 3. Load
Loads every entry in `DEPLOYMENTS`. A failure for one deployment is reported and the rest carry on.

In [ ]:
deployments = {}
for name in DEPLOYMENTS:
    try:
        deployments[name] = load_deployment(name)
        print(deployments[name])
    except Exception as err:
        print(f'{name}: FAILED to load -> {type(err).__name__}: {err}')

## 4. Working with one deployment

In [ ]:
d = deployments['ng5_2025-06-18']
d.summary()

In [ ]:
d.sensors

In [ ]:
d.retro(d.retros[0]).head()

In [ ]:
import matplotlib.pyplot as plt

rate = '60s'
mean = d.ch4(rate)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [2.2, 1]})

ax = axes[0]
for retro in mean.columns:
    series = mean[retro].dropna()
    ax.plot(series.index, series.values, marker='.', lw=1, label=retro)
ax.set_ylabel('CH$_4$ (ppm)')
ax.set_title(f'{d.name}: CH$_4$ per retro ({rate} means)')
ax.legend(ncol=4, fontsize=8)

ax = axes[1]
s = d.sensors
for retro, row in s.iterrows():
    ax.plot([0, row['east']], [0, row['north']], color='0.7', lw=1)
    ax.annotate(retro, (row['east'], row['north']), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.scatter(s['east'], s['north'], s=25, zorder=3)
ax.scatter([0], [0], marker='^', s=80, color='k', zorder=3, label='Orion')
ax.set_aspect('equal')
ax.set_xlabel('east (m)')
ax.set_ylabel('north (m)')
ax.set_title('Beam geometry')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
d.wind.describe()

## 5. Across deployments

In [ ]:
pd.concat({name: dep.summary() for name, dep in deployments.items()}, names=['deployment'])

## 6. Handing data to the history-matching notebook
`d.data` is the same `MiricoData` object the history-matching notebook calls `liv_data`, and `d.obs_targets('300s')` reproduces its `obs_targets`. For example:

```python
liv_data = d.data
obs_targets = d.obs_targets('300s')
```

In [ ]:
obs_targets = d.obs_targets('300s')
print(len(obs_targets), 'targets, e.g.', next(iter(obs_targets.items())))